# Coffee Futures — Reproducible Quantitative Analysis Notebook

**Independent Citi/Forage educational simulation**, not observed market quotations. This notebook walks through core calculations. The complete pipeline can also be reproduced using `python scripts/run_analysis.py`.

> Run all cells in order after installing `requirements.txt`.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'data' / 'inputs.json').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
from coffee_quant.models import futures_cost_of_carry, black_scholes_spot_call, black76_futures_call
from coffee_quant.simulation import risk_neutral_terminal, sample_price_paths
from coffee_quant.risk import parametric_var, expected_credit_loss, procurement_costs
from coffee_quant.note import structured_note_payoff, note_budget
import numpy as np
import matplotlib.pyplot as plt
p = json.loads((ROOT / 'data' / 'inputs.json').read_text(encoding='utf-8'))
p


## 1. Futures pricing and European call valuation

Cost-of-carry: $F_0=S_0e^{(r+u-y)T}$. **Black-76** is used for a European option on futures. The learning guide's spot Black-Scholes formula is retained for comparison.

In [ ]:
S=p['spot_price'];r=p['risk_free_rate_annual'];u=p['storage_cost_rate_annual'];y=p['convenience_yield_annual'];T=p['maturity_years'];K=p['strike_price'];sig=p['volatility_annual']
F=futures_cost_of_carry(S,r,u,T,y)
spot_call=black_scholes_spot_call(S,K,r,T,sig)
futures_call=black76_futures_call(F,K,r,T,sig)
print(f'Futuro: {F:.5f} USD/lb; BS spot: {spot_call:.5f}; Black-76: {futures_call:.5f}')


## 2. Risk-neutral Monte Carlo

The GBM terminal price simulation numerically checks the option value. It **does not forecast real-world coffee prices** six months ahead.

In [ ]:
mc=risk_neutral_terminal(S,r,u,T,sig,K,n=p['simulation_n'],seed=p['simulation_seed'],convenience_yield=y)
print(f'Terminal mean: {mc.mean_terminal_price:.5f}; MC premium: {mc.call_price:.5f}; Black-76: {futures_call:.5f}')
print(f'95% Monte Carlo confidence interval: [{mc.ci_95_low:.5f}, {mc.ci_95_high:.5f}]')


In [ ]:
fig,ax=plt.subplots(figsize=(9,3.5))
ax.hist(mc.terminal_prices,bins=80,density=True,alpha=.7)
ax.axvline(F,ls='--',color='black',label='Theoretical futures')
ax.set(xlabel='USD / lb',ylabel='Density',title='Simulated GBM terminal coffee price distribution')
ax.legend();plt.show()


## 3. Hedging a coffee purchaser (illustrative example)

An educational exposure of **100,000 lb** can be hedged with long futures or long calls. Terminal prices below are scenario inputs, not market observations.

In [ ]:
qty=p['notional_lb']
for spotT in (0.96,1.20,1.44):
    outcome=procurement_costs(spotT,qty,F,K,futures_call)
    print(spotT, {name:round(val,2) for name,val in outcome.items()})


## 4. Structured note payoff and theoretical budget

Maturity payments are contractual and remain subject to issuer credit risk. Not all real-world costs are included.

In [ ]:
note=p['structured_note']
for st in (0.95,1.20,1.25,1.40,1.60):
    print(st, round(structured_note_payoff(st,note['principal'],note['fixed_coupon_rate_six_months'],note['upside_participation'],K),2))
print(note_budget(note['principal'],note['fixed_coupon_rate_six_months'],note['upside_participation'],K,futures_call,r,T))


## 5. Value at Risk and expected credit loss

Only instructional parameters and explicitly analyst-created examples are used. No empirical historical risk is being measured.

In [ ]:
var=parametric_var(S*qty,sig,.95,1/12)
credit=p['supplier_credit_example']
el=expected_credit_loss(credit['probability_of_default'],credit['loss_given_default'],credit['exposure_at_default_usd'])
print(f'One-month 95% VaR: USD {var:,.2f}')
print(f'Illustrative expected credit loss: USD {el:,.2f}')


## 6. Limitations and next steps

- Reproducible simulation: 200,000 scenarios and a fixed random seed.
- **No actual market history, observed supplier defaults, or out-of-sample backtests** are present.
- Before real deployment: obtain executable futures/option curves, bid-ask spreads, implied volatilities, contract specifications, basis and liquidity, funding costs, Greeks, stress calibration, counterparty approvals, and independent model validation.